# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

c:\Users\rom1c\anaconda3\envs\bigdata\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [5]:
from pyspark.sql import functions as F

# Add a unique identifier column for each row
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())

# Verify the new column
df_trips.select("trip_id", "tpep_pickup_datetime", "trip_distance", "total_amount").show(5)

+-----------+--------------------+-------------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|total_amount|
+-----------+--------------------+-------------+------------+
|25769803776| 2019-01-01 00:46:40|          1.5|        9.95|
|25769803777| 2019-01-01 00:59:47|          2.6|        16.3|
|25769803778| 2018-12-21 13:48:30|          0.0|         5.8|
|25769803779| 2018-11-28 15:52:25|          0.0|        7.55|
|25769803780| 2018-11-28 15:56:57|          0.0|       55.55|
+-----------+--------------------+-------------+------------+
only showing top 5 rows


In [6]:
# Find the trip with the highest passenger count
highest_passenger_trip = (
    df_trips.select("trip_id", "passenger_count")
    .orderBy(F.col("passenger_count").desc())
    .first()
)

print(f"Highest passenger count: {highest_passenger_trip['passenger_count']} (Trip ID: {highest_passenger_trip['trip_id']})")

# Calculate the average passenger count across all trips
avg_passenger_count = df_trips.select(F.avg("passenger_count")).first()[0]
print(f"Average passenger count: {avg_passenger_count:.2f}")

Highest passenger count: 9.0 (Trip ID: 25770753732)
Average passenger count: 1.57


In [8]:
# Compute trip duration in minutes
# Note: Cast to timestamp first to handle TIMESTAMP_NTZ data types properly
df_trips = df_trips.withColumn(
    "duration_min",
    (
        F.col("tpep_dropoff_datetime").cast("timestamp").cast("long") 
        - F.col("tpep_pickup_datetime").cast("timestamp").cast("long")
    ) / 60
)

# Filter out obvious invalid values (negative distances or durations)
df_clean_trips = df_trips.filter((F.col("trip_distance") >= 0) & (F.col("duration_min") >= 0))

# Shortest and longest trip by distance
shortest_dist = df_clean_trips.orderBy(F.col("trip_distance").asc()).select("trip_id", "trip_distance").first()
longest_dist = df_clean_trips.orderBy(F.col("trip_distance").desc()).select("trip_id", "trip_distance").first()

print(f"Shortest distance: {shortest_dist['trip_distance']} miles (Trip ID: {shortest_dist['trip_id']})")
print(f"Longest distance: {longest_dist['trip_distance']} miles (Trip ID: {longest_dist['trip_id']})")

# Shortest and longest trip by duration
shortest_duration = df_clean_trips.orderBy(F.col("duration_min").asc()).select("trip_id", "duration_min").first()
longest_duration = df_clean_trips.orderBy(F.col("duration_min").desc()).select("trip_id", "duration_min").first()

print(f"Shortest duration: {shortest_duration['duration_min']:.2f} mins (Trip ID: {shortest_duration['trip_id']})")
print(f"Longest duration: {longest_duration['duration_min']:.2f} mins (Trip ID: {longest_duration['trip_id']})")

Shortest distance: 0.0 miles (Trip ID: 25769803778)
Longest distance: 831.8 miles (Trip ID: 25775877867)
Shortest duration: 0.00 mins (Trip ID: 25769803804)
Longest duration: 43648.02 mins (Trip ID: 25769872043)


In [9]:
# Filter specifically for January 2019 to remove rogue date records
df_jan_2019 = df_trips.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01") & 
    (F.col("tpep_pickup_datetime") < "2019-02-01")
)

# Count trips per calendar day
daily_trips = (
    df_jan_2019.groupBy(F.to_date("tpep_pickup_datetime").alias("pickup_date"))
    .count()
    .orderBy("count")
)

slowest_day = daily_trips.first()
busiest_day = daily_trips.orderBy(F.col("count").desc()).first()

print(f"Slowest day: {slowest_day['pickup_date']} with {slowest_day['count']} trips")
print(f"Busiest day: {busiest_day['pickup_date']} with {busiest_day['count']} trips")

Slowest day: 2019-01-01 with 189432 trips
Busiest day: 2019-01-25 with 292499 trips


In [10]:
# Extract the pickup hour (0 to 23)
df_trips = df_trips.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

# Aggregate trip counts by hour
hourly_trips = df_trips.groupBy("pickup_hour").count().orderBy("count")

slowest_hour = hourly_trips.first()
busiest_hour = hourly_trips.orderBy(F.col("count").desc()).first()

print(f"Slowest hour: {slowest_hour['pickup_hour']}:00 with {slowest_hour['count']} trips")
print(f"Busiest hour: {busiest_hour['pickup_hour']}:00 with {busiest_hour['count']} trips")

Slowest hour: 4:00 with 61424 trips
Busiest hour: 18:00 with 515390 trips


In [11]:
# Group by day of week (1 = Sunday, 2 = Monday, ..., 7 = Saturday)
dow_summary = (
    df_jan_2019.groupBy(
        F.dayofweek("tpep_pickup_datetime").alias("day_of_week_num"),
        F.date_format("tpep_pickup_datetime", "EEEE").alias("day_name")
    )
    .count()
    .orderBy("count")
)

dow_summary.show()

slowest_dow = dow_summary.first()
busiest_dow = dow_summary.orderBy(F.col("count").desc()).first()

print(f"Slowest day of the week: {slowest_dow['day_name']} ({slowest_dow['count']} trips)")
print(f"Busiest day of the week: {busiest_dow['day_name']} ({busiest_dow['count']} trips)")

+---------------+---------+-------+
|day_of_week_num| day_name|  count|
+---------------+---------+-------+
|              1|   Sunday| 859890|
|              2|   Monday| 907764|
|              7| Saturday|1009979|
|              6|   Friday|1087150|
|              3|  Tuesday|1209076|
|              4|Wednesday|1265229|
|              5| Thursday|1356992|
+---------------+---------+-------+

Slowest day of the week: Sunday (859890 trips)
Busiest day of the week: Thursday (1356992 trips)


In [12]:
# Calculate Pearson correlation coefficient between features and tip_amount
correlations = df_trips.select(
    F.corr("trip_distance", "tip_amount").alias("distance_vs_tip"),
    F.corr("passenger_count", "tip_amount").alias("passengers_vs_tip")
).first()

print(f"Correlation (Distance vs Tip): {correlations['distance_vs_tip']:.4f}")
print(f"Correlation (Passengers vs Tip): {correlations['passengers_vs_tip']:.4f}")

Correlation (Distance vs Tip): 0.5269
Correlation (Passengers vs Tip): 0.0011


In [13]:
# Find the record with the maximum extra charge
highest_extra_trip = (
    df_trips.select("trip_id", "extra", "total_amount")
    .orderBy(F.col("extra").desc())
    .first()
)

print(f"Highest extra charge: ${highest_extra_trip['extra']} (Trip ID: {highest_extra_trip['trip_id']})")

Highest extra charge: $535.38 (Trip ID: 25775127259)


### Data Anomalies and Outliers Observed

1. **Incorrect Timestamps / Date Leakage**: Some records have pickup dates outside of January 2019 (e.g., records from 2018 or far into the future due to misconfigured taxi meters).
2. **Zero or Negative Values**:
   - Several trips have a distance of `0.0` miles but still record fares and tips.
   - Durations computed as `<= 0` minutes where pickup and dropoff happen simultaneously or dropoff is earlier than pickup.
   - Negative amounts in `fare_amount` or `extra`, likely representing cancellations, refunds, or system adjustments.
3. **Unrealistic Trip Durations / Distances**: Extremely high trip durations (hundreds of hours) indicating trips that were never properly closed in the meter.
4. **Zero Passenger Records**: Records with `passenger_count = 0`, which might be delivery services, erroneous entries, or off-duty runs.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing